In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score
from xgboost import XGBClassifier

In [ ]:
full_grid_encoded = pd.read_csv("../data/taylor_swift_model_df_final_nlp.csv")

In [ ]:

tour_order = [
    "Fearless", "Speak Now World Tour", "The Red Tour",
    "The 1989 World Tour", "reputation Stadium Tour", "The Eras Tour"
]

feature_cols_nlp = [
    "played_last_tour", "song_age_tours", "tour_type_single_album",
    "sentiment", "avg_sentence_len", "proper_noun_density", "unique_word_ratio"
]

target_col = "was_played_on_tour"

In [ ]:
results = {}
for held_out_tour in tour_order:
    train = full_grid_encoded[full_grid_encoded["tour"] != held_out_tour]
    test = full_grid_encoded[full_grid_encoded["tour"] == held_out_tour]
    
    X_train, y_train = train[feature_cols_nlp], train[target_col]
    X_test, y_test = test[feature_cols_nlp], test[target_col]
    
    model = XGBClassifier(eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, probs)
    results[held_out_tour] = pr_auc
    print(f"Held out: {held_out_tour:30s} PR-AUC: {pr_auc:.3f}")

print("\nAverage PR-AUC:", sum(results.values()) / len(results))

Held out: Fearless                       PR-AUC: 1.000
Held out: Speak Now World Tour           PR-AUC: 0.990
Held out: The Red Tour                   PR-AUC: 0.810
Held out: The 1989 World Tour            PR-AUC: 0.932
Held out: reputation Stadium Tour        PR-AUC: 0.748
Held out: The Eras Tour                  PR-AUC: 0.957

Average PR-AUC: 0.9061839694705509


In [ ]:
model_full_nlp = XGBClassifier(eval_metric="logloss", random_state=42)
model_full_nlp.fit(full_grid_encoded[feature_cols_nlp], full_grid_encoded[target_col])

importances = pd.Series(model_full_nlp.feature_importances_, index=feature_cols_nlp).sort_values(ascending=False)
print(importances)

song_age_tours            0.790173
tour_type_single_album    0.120531
played_last_tour          0.089296
is_new_album_song         0.000000
dtype: float32


In [ ]:
import joblib
joblib.dump(model_full_nlp, "../data/xgb_model_final_nlp.pkl")